# Nepali OCR VLM — Kaggle T4 Training Notebook

**Architecture**: ViT-S/16 (448×448) → Connector → GDN/Transformer Hybrid Decoder  
**Output**: Structured HTML (paragraphs, headings, tables)  
**GPU**: T4 16GB  

## Training order
1. Language pretrain on Nepali Wikipedia (decoder only, ~6h)
2. OCR fine-tune on English docs + Nepali synthetic (~4h each)

In [ ]:
# ── 1. Clone repo and install deps ────────────────────────────────────────────
!git clone https://github.com/YOUR_USERNAME/nepali-ocr-vlm.git
%cd nepali-ocr-vlm
!pip install -q transformers datasets torch torchvision opencv-python-headless pillow

In [ ]:
# ── 2. Download Noto Devanagari font ──────────────────────────────────────────
import os
os.makedirs('fonts', exist_ok=True)

# Option A: from Google Fonts (requires wget)
!wget -q 'https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansDevanagari/NotoSansDevanagari-Regular.ttf' \
     -O fonts/NotoSansDevanagari-Regular.ttf

# Verify
assert os.path.exists('fonts/NotoSansDevanagari-Regular.ttf'), 'font download failed'
print('Font ready ✅')

In [ ]:
# ── 3. Prepare data ───────────────────────────────────────────────────────────
# Downloads Nepali Wikipedia corpus + RenderedText + CORD + IAM
# Then generates synthetic Nepali document images
# Total: ~5-10GB, ~30min on T4

!PYTHONPATH=. python scripts/prepare_data.py --stage all --max_samples 100000

!PYTHONPATH=. python scripts/prepare_data.py \
    --stage synth \
    --font_path fonts/NotoSansDevanagari-Regular.ttf \
    --n_synth 50000

!PYTHONPATH=. python scripts/prepare_data.py --stage merge

In [ ]:
# ── 4. Verify GPU and model size ──────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

import sys; sys.path.insert(0, '.')
from src.models.vlm import NepaliOCR, DEFAULT_CONFIG
model = NepaliOCR(**DEFAULT_CONFIG)
print(model.param_summary())

In [ ]:
# ── 5a. Stage 1: Language pretrain ────────────────────────────────────────────
# ~6 hours on T4. Trains only the HybridDecoder on Nepali text.
# Skip if you already have checkpoints/pretrain.pt

!PYTHONPATH=. python scripts/pretrain.py \
    --config configs/pretrain.json \
    --train_path data/corpus/train.jsonl \
    --dev_path   data/corpus/val.jsonl \
    --device cuda

In [ ]:
# ── 5b. Stage 2: OCR fine-tune on English docs ────────────────────────────────
# Trains the full VLM on RenderedText + CORD + IAM
# Model learns to output structured HTML from document images

!PYTHONPATH=. python scripts/train_ocr.py \
    --config       configs/ocr_finetune.json \
    --pretrain_ckpt checkpoints/pretrain.pt \
    --train_path   data/documents/train.jsonl \
    --dev_path     data/documents/val.jsonl \
    --device       cuda

In [ ]:
# ── 5c. Stage 3: Fine-tune on Nepali synthetic ────────────────────────────────
# Lower LR (3e-5) — fine-tuning a model that already reads English docs
# to now read Devanagari script

!PYTHONPATH=. python scripts/train_ocr.py \
    --config        configs/ocr_finetune.json \
    --pretrain_ckpt checkpoints/ocr.pt \
    --train_path    data/documents/nepali_synth/data_train.jsonl \
    --dev_path      data/documents/nepali_synth/data_val.jsonl \
    --lr            3e-5 \
    --train_steps   10000 \
    --out_ckpt      checkpoints/ocr_nepali.pt \
    --device        cuda

In [ ]:
# ── 6. Run inference ──────────────────────────────────────────────────────────
from src.detection.pipeline import NepaliOCRPipeline
from PIL import Image

pipeline = NepaliOCRPipeline.from_checkpoint('checkpoints/ocr_nepali.pt')

# Test on a sample image
result = pipeline.run('your_document.png')
print(result.full_text)